# A small scaling experiment

**Track:** Training LLMs. **Level:** Intermediate. **CPU:** yes.

## Goals

- Train the tiny GPT at three widths on the same data and the same step count.
- Train one width on a quarter of the data.
- Fit a line to log loss versus log parameter count, and say what the line is not.

## Sources and license

Original experiment. The questions come from Kaplan et al., *Scaling Laws for Neural Language Models* (OpenAI, 2020) and Hoffmann et al., *Training Compute-Optimal Large Language Models* (Chinchilla, DeepMind, 2022). No figure or code from those papers is reproduced. The model is the tiny GPT from the previous lab.

## Concepts

A scaling law, in those papers, is a power-law fit of loss against model size, data, or compute, measured at convergence or at a compute-optimal stop. This lab is smaller than that in every way. The step count is fixed, so a wider model also uses more compute per step. The data is a repeated sentence, so "more data" means more copies of the same pattern, not more diversity. The plot is still the right shape of experiment: change one axis, record the loss, and do not claim a law the design cannot support.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'numpy', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from labs._shared.seed import seed_everything

seed_everything(0)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        b, t, c = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        mixed = (weights @ v).transpose(1, 2).contiguous().view(b, t, c)
        return self.proj(mixed)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    """A nanoGPT-style decoder. Original code, same block layout as nanoGPT (MIT)."""

    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size) for _ in range(n_layer)]
        )
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None, loss_mask=None):
        b, t = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(t, device=idx.device))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            per = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                reduction="none",
            ).view(b, t)
            if loss_mask is None:
                loss = per.mean()
            else:
                loss = (per * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_new, temperature=0.8):
        self.eval()
        for _ in range(n_new):
            logits, _ = self(idx[:, -self.block_size :])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def train_lm(data, vocab_size, steps, n_embd=64, n_layer=2, n_head=4, block_size=32, batch_size=32, lr=3e-3, seed=0):
    g = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = TinyGPT(vocab_size, n_embd=n_embd, n_head=n_head, n_layer=n_layer, block_size=block_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i : i + block_size] for i in ix])
        y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(float(loss.detach()))
    return model, losses

text = ("the cat sat on the mat. the cat sat on the hat. the dog sat on the log. " * 50)
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
full = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
quarter = full[: max(200, len(full) // 4)]
print("tokens", len(full), "quarter", len(quarter), "vocab", len(chars))


## Guided exercise: width and data

Parameter count is measured, not assumed. The compute proxy is `steps * layers * width^2`. It is not FLOPs. It is enough to show that the widest run did more work per step.


In [ ]:
def final_loss(data, n_embd, n_layer, steps=40, seed=0):
    model, losses = train_lm(
        data,
        vocab_size=len(chars),
        steps=steps,
        n_embd=n_embd,
        n_layer=n_layer,
        n_head=4,
        block_size=32,
        batch_size=32,
        lr=3e-3,
        seed=seed,
    )
    n_params = sum(p.numel() for p in model.parameters())
    return float(np.mean(losses[-5:])), n_params, float(np.mean(losses[:5]))

rows = []
for n_embd, n_layer in ((32, 1), (64, 2), (96, 2)):
    end, n_params, start = final_loss(full, n_embd, n_layer, seed=0)
    rows.append({
        "width": n_embd,
        "layers": n_layer,
        "params": n_params,
        "loss": end,
        "start": start,
        "data": len(full),
        "compute_proxy": 40 * n_layer * n_embd ** 2,
    })
end_q, params_q, start_q = final_loss(quarter, 64, 2, seed=0)
rows.append({
    "width": 64, "layers": 2, "params": params_q, "loss": end_q, "start": start_q,
    "data": len(quarter), "compute_proxy": 40 * 2 * 64 ** 2,
})
for row in rows:
    print(
        f"width {row['width']:3d} layers {row['layers']} data {row['data']:4d} "
        f"params {row['params']:7d} loss {row['loss']:.3f}"
    )
size_rows = [row for row in rows if row["data"] == len(full)]
assert size_rows[-1]["loss"] < size_rows[0]["loss"]
assert size_rows[-1]["compute_proxy"] > size_rows[0]["compute_proxy"]
full_loss = next(row["loss"] for row in rows if row["width"] == 64 and row["data"] == len(full))
assert full_loss <= end_q + 0.05

log_params = np.log(np.array([row["params"] for row in size_rows], dtype=float))
log_loss = np.log(np.array([row["loss"] for row in size_rows], dtype=float))
slope, intercept = np.polyfit(log_params, log_loss, 1)
print(f"log(loss) ~ {slope:.3f} * log(params) + {intercept:.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(log_params, log_loss)
xs = np.linspace(log_params.min(), log_params.max(), 20)
ax.plot(xs, slope * xs + intercept)
ax.set_xlabel("log parameters")
ax.set_ylabel("log final loss")
ax.set_title("Fixed steps, repeated sentence")
plt.tight_layout()
plt.show()
print("LAB OK")


## Your turn

The quarter-data run uses the medium model. Print its loss next to the full-data medium model and the ratio of their token counts.

## Solution


In [ ]:
print("full", round(full_loss, 3), "quarter", round(end_q, 3), "token ratio", round(len(full) / len(quarter), 2))
assert len(full) > len(quarter)


## Beyond the course

Chinchilla's claim is about the loss at the compute-optimal allocation, not at a shared step count. Recompute the proxy and state the comparison this lab cannot make: we did not match compute across widths, and we did not train to a loss plateau. The slope above is a description of these three runs.


In [ ]:
print("compute proxy by width", [(row["width"], row["compute_proxy"]) for row in size_rows])
print("not a Chinchilla frontier: step count is shared, so proxy grows with width**2")
assert all(row["loss"] < row["start"] for row in size_rows)


## Research questions

1. The widest model finished lower. How much of that is extra parameters, and how much is the extra compute in the proxy column?
2. The quarter corpus is a prefix of the same repeated sentence. What would "less data" mean if the corpus were a real mixture of documents?
3. Kaplan et al. fit loss against compute with a power law on models that are orders of magnitude larger. Which assumption of that fit is visibly false on three points?

## Further reading

- Kaplan et al., *Scaling Laws for Neural Language Models* (OpenAI, 2020).
- Hoffmann et al., *Training Compute-Optimal Large Language Models* (DeepMind, 2022).
- Bahri et al., *Explaining Neural Scaling Laws* (2021).
